## Structured Data Cleaning

## Purpose
Clean persistent structured Delta/CSV assets into trusted ClickHouse tables with auditable schema, quality, and rejection evidence.

## Inputs
Persistent structured landing assets under `landing-zone/persistent-landing/structured/raw/`, including global warming, temperature change, vehicle emissions, and natural disaster tweets.

## Validation / quality checks
The notebook checks required columns, row counts, duplicate keys, encoding/string quality, dataset-specific rules, and rejected-row evidence where validation fails.

## Transformation / cleaning logic
Cleaning normalizes table and column names, applies dataset-specific fixes, removes invalid required rows, deduplicates records, and writes trusted ClickHouse tables.

## Metadata and lineage fields
Trusted outputs carry fields such as `source_file_path`, `schema_version`, `validation_status`, `ingestion_time`, owner/steward/classification fields, and RBAC role evidence.

## Output assets
`bi_analytics.global_warming_dataset`, `bi_analytics.temperature_change`, `bi_analytics.co2_emission_by_vehicles`, `bi_analytics.natural_disaster_tweets`, and trusted rejected evidence under `trusted-zone/rejected/`.

## RBAC / service user used
The production DAG uses ClickHouse trusted writer/reader and MinIO trusted/landing service users configured through environment variables.

## Notebook-DAG alignment note
This notebook mirrors the trusted structured cleaning logic implemented for `trusted_zone`; it does not import DAG functions and is kept as report evidence for the same business rules.


## Execution Notes and Batch Logging
This notebook cleans structured Landing Zone datasets into trusted ClickHouse tables. Dataset paths are logged as cleaning batches, ClickHouse sync reports Spark partition batches, and each processing result prints status and row counts.


**Architecture Overview:**
This notebook executes an industrial-grade data cleaning and synchronization task. We read Parquet/Delta files directly from the MinIO (S3) data lake using Apache Spark, perform deep structural cleaning in distributed memory (primary key hardening, anomaly handling, dynamic field inference), and finally synchronize the data to the ClickHouse modern analytical data warehouse via high-concurrency direct writes (MapPartitions).

**Pipeline Steps:**
1. **Environment Initialization**: Mount the Spark Session and connect to MinIO and ClickHouse.
2. **Dynamic Metadata Scanning**: Automatically identify valid data directories in the S3 Landing Zone.
3. **Distributed Adaptive Cleaning**: Apply different deduplication rules for different business tables (e.g., full-field deduplication for vehicle tables, composite primary key deduplication for climate tables).
4. **High-Concurrency Ingestion**: Pre-build DDL on the Driver, and perform distributed direct writes to ClickHouse from the Executors.
5. **Post-Cleaning Validation**: Check physical metrics, data distribution, and real sample exploration.

**Production synchronization note: structured cleaning rules and quarantine.**

The production Trusted Zone DAG declares dataset-specific quality rules for `co2_emission_by_vehicles`, `global_warming_dataset`, `natural_disaster_tweets`, and `temperature_change`: expected columns, required columns, type/range normalization, deduplication behavior, and per-dataset schema versions. Rows or datasets that fail required checks are excluded from trusted ClickHouse business tables and persisted as rejected/quarantine evidence under `s3://trusted-zone/rejected/structured/`. This notebook mirrors the same rules and rejected-event structure inline for transparent, step-by-step review without importing DAG helper code.


**Importing Useful Libraries**

In [1]:
import os
import re
import ast
import boto3
import clickhouse_connect
from datetime import datetime, timezone
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, DateType, TimestampType, ArrayType
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "writer"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")
TRUSTED_LOGICAL_DATE = os.getenv("TRUSTED_LOGICAL_DATE") or datetime.now(timezone.utc).isoformat()

# 2. Initialize Spark Session with S3 & Delta bindings
DELTA_VERSION = "4.1.0" 

spark = SparkSession.builder \
    .appName("Production-Data-Warehouse-Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            f"org.apache.hadoop:hadoop-aws:3.3.4,"
            f"com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            f"io.delta:delta-spark_2.13:{DELTA_VERSION}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# 3. Normalize Hadoop config values to prevent JVM parse errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key, value = item.getKey(), item.getValue()
    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        hadoop_conf.set(key, "".join(char for char in value if char.isdigit()))

print("Spark Cluster environment initialized successfully.")

Spark Cluster environment initialized successfully.


In [2]:
STRUCTURED_BASE_PATH = "s3a://landing-zone/persistent-landing/structured/"
EXCLUDED_DATASET_FOLDERS = {"raw", "file_catalog"}


def discover_structured_dataset_paths(spark_session: SparkSession, base_path: str = STRUCTURED_BASE_PATH) -> list[str]:
    """Return valid structured dataset directories from the landing-zone path."""
    sc = spark_session.sparkContext
    path_obj = sc._jvm.org.apache.hadoop.fs.Path(base_path)
    fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

    dataset_paths = []
    for status in fs.listStatus(path_obj):
        if not status.isDirectory():
            continue

        full_path = status.getPath().toString()
        folder_name = full_path.rstrip("/").split("/")[-1]
        if folder_name in EXCLUDED_DATASET_FOLDERS or folder_name.startswith("."):
            continue

        dataset_paths.append(full_path)

    return sorted(dataset_paths)


valid_delta_paths = discover_structured_dataset_paths(spark)
print(f"Discovered {len(valid_delta_paths)} analytical dataset targets in S3 Landing Zone.")
for batch_no, dataset_path in enumerate(valid_delta_paths, start=1):
    print(f"[structured discovery batch {batch_no}/{len(valid_delta_paths)}] {dataset_path}")


Discovered 4 analytical dataset targets in S3 Landing Zone.
[structured discovery batch 1/4] s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975
[structured discovery batch 2/4] s3a://landing-zone/persistent-landing/structured/global_warming_dataset_1780764136045
[structured discovery batch 3/4] s3a://landing-zone/persistent-landing/structured/natural_disaster_tweets_1780764519520
[structured discovery batch 4/4] s3a://landing-zone/persistent-landing/structured/temperature_change_1780764519800


### Step 1 - Declare Dataset-Specific Quality Rules

This notebook mirrors the production structured-cleaning rules inline, without importing DAG code. The table below makes cleaning correctness explicit for each structured dataset before any data is written to ClickHouse.


In [3]:
from functools import reduce
import hashlib
import json
from pyspark.sql.types import BooleanType, ShortType, ByteType, DecimalType
import pandas as pd

NULL_TOKENS = {"", "na", "n/a", "null", "none", "nan", "-", "--"}
DEGREE_C = chr(0x00b0) + "C"
CORRUPTED_DEGREE_C_PATTERN = f"(?i)[{chr(0x00e2)}{chr(0x00c2)}]\\s*[{chr(0x00b0)}{chr(0x00ba)}]\\s*c"
CLEAN_DEGREE_C_PATTERN = f"(?i){chr(0x00b0)}\\s*c"
MOJIBAKE_DASH_PATTERN = f"{chr(0x00e2)}{chr(0x0080)}[{chr(0x0093)}{chr(0x0094)}]"
ARRAY_LIKE_COLUMNS = {"hashtags", "emojis"}

TABLE_SORTING_KEYS = {
    "global_warming": ["country", "year"],
    "temperature_change": ["area", "months", "year"],
    "emission": ["make", "model", "vehicle_class"],
    "tweet": ["id"],
}

TRUSTED_SCHEMA_VERSION = "trusted_v1"
STRUCTURED_DATASET_RULES = {
    "co2_emission_by_vehicles": {
        "schema_version": "co2_emission_by_vehicles_v1",
        "required_columns": ["make", "model", "vehicle_class"],
        "business_keys": ["make", "model", "vehicle_class", "transmission", "fuel_type"],
        "expected_columns": [
            "make", "model", "vehicle_class", "engine_sizel", "cylinders", "transmission", "fuel_type",
            "fuel_consumption_city_l_100_km", "fuel_consumption_hwy_l_100_km", "fuel_consumption_comb_l_100_km",
            "fuel_consumption_comb_mpg", "co2_emissionsg_km",
        ],
        "integer_columns": ["cylinders", "fuel_consumption_comb_mpg", "co2_emissionsg_km"],
        "non_negative_columns": [
            "engine_sizel", "cylinders", "fuel_consumption_city_l_100_km", "fuel_consumption_hwy_l_100_km",
            "fuel_consumption_comb_l_100_km", "fuel_consumption_comb_mpg", "co2_emissionsg_km",
        ],
    },
    "global_warming_dataset": {
        "schema_version": "global_warming_dataset_v1",
        "required_columns": ["country", "year"],
        "business_keys": ["country", "year"],
        "expected_columns": ["country", "year", "temperature_anomaly", "co2_emissions", "population"],
        "integer_columns": ["year", "extreme_weather_events"],
        "non_negative_columns": ["co2_emissions", "population", "forest_area", "gdp", "methane_emissions"],
        "year_range": (1900, 2100),
    },
    "natural_disaster_tweets": {
        "schema_version": "natural_disaster_tweets_v1",
        "required_columns": ["id"],
        "business_keys": ["id"],
        "unique_keys": ["id"],
        "expected_columns": ["id", "text", "label", "hashtags", "emojis"],
    },
    "temperature_change": {
        "schema_version": "temperature_change_v1",
        "required_columns": ["area", "months", "year"],
        "business_keys": ["area", "months", "year"],
        "expected_columns": [
            "domain_code", "domain", "area_code_m49", "area", "element_code", "element", "months_code", "months",
            "year_code", "year", "unit", "value", "flag", "flag_description",
        ],
        "integer_columns": ["year", "year_code"],
        "year_range": (1961, 2100),
    },
}


def rules_for_table(table_name: str) -> dict:
    for table_hint, rules in STRUCTURED_DATASET_RULES.items():
        if table_hint in table_name:
            return rules
    return {"schema_version": TRUSTED_SCHEMA_VERSION, "required_columns": [], "business_keys": [], "expected_columns": []}


def required_columns_for_table(table_name: str) -> list[str]:
    return list(rules_for_table(table_name).get("required_columns", []))


def schema_version_for_table(table_name: str) -> str:
    return rules_for_table(table_name).get("schema_version", TRUSTED_SCHEMA_VERSION)


rule_rows = []
for dataset_name, rules in STRUCTURED_DATASET_RULES.items():
    rule_rows.append({
        "dataset": dataset_name,
        "schema_version": rules.get("schema_version"),
        "required_columns": ", ".join(rules.get("required_columns", [])),
        "business_keys": ", ".join(rules.get("business_keys", [])),
        "type_casts": ", ".join(rules.get("integer_columns", [])) or "-",
        "range_rules": f"year {rules['year_range']}" if "year_range" in rules else "non-negative numeric masking where configured",
        "rejected_condition": "missing required columns or missing required row values",
    })
display(pd.DataFrame(rule_rows))


,dataset,schema_version,required_columns,business_keys,type_casts,range_rules,rejected_condition
0,co2_emission_by_vehicles,co2_emission_by_vehicles_v1,"make, model, vehicle_class","make, model, vehicle_class, transmission, fuel...","cylinders, fuel_consumption_comb_mpg, co2_emis...",non-negative numeric masking where configured,missing required columns or missing required r...
1,global_warming_dataset,global_warming_dataset_v1,"country, year","country, year","year, extreme_weather_events","year (1900, 2100)",missing required columns or missing required r...
2,natural_disaster_tweets,natural_disaster_tweets_v1,id,id,-,non-negative numeric masking where configured,missing required columns or missing required r...
3,temperature_change,temperature_change_v1,"area, months, year","area, months, year","year, year_code","year (1961, 2100)",missing required columns or missing required r...


### Step 2 - Source Reading and Schema Normalization

The next helpers discover stable table names, read Delta or Parquet sources, normalize headers, and report expected-schema gaps. Missing expected columns are non-blocking evidence; missing required columns are blocking and go to rejected evidence.


In [4]:
CLICKHOUSE_TYPE_MAP = {
    StringType: "String",
    IntegerType: "Int32",
    LongType: "Int64",
    ShortType: "Int16",
    ByteType: "Int8",
    FloatType: "Float32",
    DoubleType: "Float64",
    BooleanType: "UInt8",
    DateType: "Date",
    TimestampType: "DateTime",
}


def normalize_table_name(s3_path: str) -> str:
    folder_name = s3_path.rstrip("/").split("/")[-1]
    if folder_name in {"raw", "_staging", "file_catalog"} or folder_name.startswith("."):
        return ""
    table_name = folder_name.replace("_delta", "")
    table_name = re.sub(r"_\d+$", "", table_name)
    table_name = re.sub(r"_\d{8}t\d{6}$", "", table_name, flags=re.IGNORECASE)
    table_name = re.sub(r"[^0-9A-Za-z_]+", "_", table_name)
    return re.sub(r"_+", "_", table_name).strip("_").lower()


def read_delta_or_parquet(spark_session: SparkSession, s3_path: str) -> DataFrame:
    sc = spark_session.sparkContext
    delta_log_path = sc._jvm.org.apache.hadoop.fs.Path(s3_path.rstrip("/") + "/_delta_log")
    fs = delta_log_path.getFileSystem(sc._jsc.hadoopConfiguration())
    reader_format = "delta" if fs.exists(delta_log_path) else "parquet"
    print(f"Reading {s3_path} as {reader_format}")
    return spark_session.read.format(reader_format).load(s3_path)


def normalize_column_name(column_name: str, position: int) -> str:
    cleaned = str(column_name).replace("\ufeff", "")
    cleaned = re.sub(r"[()]+", "", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z_]+", "_", cleaned.strip())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    cleaned = cleaned or f"column_{position + 1}"
    if cleaned.lower() in {"tweet_id", "id"}:
        cleaned = "id"
    if cleaned[0].isdigit():
        cleaned = f"col_{cleaned}"
    return cleaned.lower()


def standardize_column_names(df: DataFrame) -> DataFrame:
    seen = {}
    renamed_df = df
    mapping = {}
    for idx, original_name in enumerate(df.columns):
        base_name = normalize_column_name(original_name, idx)
        occurrence = seen.get(base_name, 0)
        seen[base_name] = occurrence + 1
        final_name = base_name if occurrence == 0 else f"{base_name}_{occurrence + 1}"
        mapping[original_name] = final_name
        if final_name != original_name:
            renamed_df = renamed_df.withColumnRenamed(original_name, final_name)
    print("Column normalization mapping:", mapping)
    return renamed_df


def log_expected_schema_gaps(df: DataFrame, table_name: str) -> list[str]:
    expected_columns = rules_for_table(table_name).get("expected_columns", [])
    missing = [column for column in expected_columns if column not in df.columns]
    if missing:
        print(f"Expected schema gaps for {table_name}: {missing}")
    else:
        print(f"Expected schema check passed for {table_name}")
    return missing


### Step 3 - Baseline Cleaning

These helpers perform generic trusted-zone cleaning: remove empty rows, normalize strings/null tokens/encoding, clean array-like fields, mask invalid floating values, and apply isolated table-specific fixes such as casting tweet IDs to string.


In [5]:
def normalize_string_columns(df: DataFrame, table_name: str) -> DataFrame:
    string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]
    for column_name in string_columns:
        cleaned_col = F.regexp_replace(F.col(column_name), r"[\n\r\t]", " ")
        cleaned_col = F.regexp_replace(cleaned_col, r"\s+", " ")
        cleaned_col = F.regexp_replace(cleaned_col, MOJIBAKE_DASH_PATTERN, "-")
        cleaned_col = F.regexp_replace(cleaned_col, CORRUPTED_DEGREE_C_PATTERN, DEGREE_C)
        cleaned_col = F.regexp_replace(cleaned_col, CLEAN_DEGREE_C_PATTERN, DEGREE_C)
        cleaned_col = F.regexp_replace(cleaned_col, r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F-\u009F]", "")
        trimmed_col = F.trim(cleaned_col)
        lower_trimmed_col = F.lower(trimmed_col)
        if table_name == "temperature_change" and column_name == "unit":
            normalized_col = F.when(
                lower_trimmed_col.isin("c", "celsius", "degree c", "degrees c"),
                F.lit(DEGREE_C),
            ).otherwise(trimmed_col)
        else:
            normalized_col = lower_trimmed_col
        df = df.withColumn(
            column_name,
            F.when(lower_trimmed_col.isin(*sorted(NULL_TOKENS)), F.lit(None)).otherwise(normalized_col),
        )
    return df


def drop_empty_rows(df: DataFrame) -> DataFrame:
    if not df.columns:
        return df
    non_empty_checks = []
    for field in df.schema.fields:
        col_ref = F.col(field.name)
        if isinstance(field.dataType, StringType):
            non_empty_checks.append(col_ref.isNotNull() & (F.trim(col_ref) != ""))
        else:
            non_empty_checks.append(col_ref.isNotNull())
    return df.where(reduce(lambda left, right: left | right, non_empty_checks))


def mask_invalid_numeric_values(df: DataFrame) -> DataFrame:
    for field in df.schema.fields:
        if isinstance(field.dataType, (FloatType, DoubleType)):
            as_text = F.lower(F.col(field.name).cast("string"))
            df = df.withColumn(
                field.name,
                F.when(
                    F.isnan(F.col(field.name)) | as_text.isin("infinity", "+infinity", "-infinity", "inf", "+inf", "-inf"),
                    F.lit(None),
                ).otherwise(F.col(field.name)),
            )
    return df


def normalize_array_columns(df: DataFrame) -> DataFrame:
    dtype_lookup = dict(df.dtypes)
    for column_name in ARRAY_LIKE_COLUMNS.intersection(df.columns):
        if "string" in dtype_lookup[column_name]:
            df = df.withColumn(column_name, F.split(F.regexp_replace(F.col(column_name), r'[\[\]\'"\s]', ""), ","))
        df = df.withColumn(
            column_name,
            F.expr(f"filter(transform({column_name}, x -> lower(trim(x))), x -> x IS NOT NULL AND x != '')"),
        )
    return df


def apply_table_specific_fixes(df: DataFrame, table_name: str) -> DataFrame:
    if "id" in df.columns and "tweet" in table_name:
        df = df.withColumn("id", F.col("id").cast("string"))
    return df


### Step 4 - Dataset-Specific Cleaning, Rejected Evidence, and Metadata

This step mirrors the DAG's dataset-specific casts/ranges and quarantine behavior. Rows with missing required values are excluded from trusted business tables and written as structured rejected evidence under `trusted-zone/rejected/structured/`.


In [6]:
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)
STRUCTURED_CLEANING_AUDIT = []
STRUCTURED_REJECTION_EVENTS = []


def audit_step(table_name: str, step: str, before_rows, after_rows, detail: str = "") -> None:
    STRUCTURED_CLEANING_AUDIT.append({
        "table": table_name,
        "step": step,
        "before_rows": before_rows,
        "after_rows": after_rows,
        "detail": detail,
    })
    print(f"[{table_name}] {step}: {before_rows} -> {after_rows}. {detail}")


def validate_required_columns(df: DataFrame, table_name: str) -> None:
    required_columns = required_columns_for_table(table_name)
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def apply_dataset_specific_rules(df: DataFrame, table_name: str) -> DataFrame:
    rules = rules_for_table(table_name)
    for column_name in rules.get("integer_columns", []):
        if column_name in df.columns:
            df = df.withColumn(column_name, F.col(column_name).cast("int"))
    for column_name in rules.get("non_negative_columns", []):
        if column_name in df.columns:
            df = df.withColumn(
                column_name,
                F.when(F.col(column_name).cast("double") < F.lit(0), F.lit(None)).otherwise(F.col(column_name)),
            )
    if "year_range" in rules and "year" in df.columns:
        min_year, max_year = rules["year_range"]
        df = df.withColumn(
            "year",
            F.when((F.col("year") < F.lit(min_year)) | (F.col("year") > F.lit(max_year)), F.lit(None)).otherwise(F.col("year")),
        )
    if table_name == "temperature_change" and "months" in df.columns:
        df = df.withColumn("months", F.regexp_replace(F.col("months"), r"\s*-\s*", "-"))
    return df


def write_rejected_event(table_name: str, logical_date: str, event: dict) -> None:
    payload = {
        "zone": "trusted",
        "domain": "structured",
        "dataset_name": table_name,
        "validation_status": "rejected",
        "schema_version": schema_version_for_table(table_name),
        "rejected_at": logical_date,
        **event,
    }
    digest = hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:12]
    safe_date = re.sub(r"[^0-9A-Za-z]+", "_", logical_date).strip("_")
    key = f"rejected/structured/{table_name}/{safe_date}_{digest}.json"
    s3.put_object(
        Bucket="trusted-zone",
        Key=key,
        Body=json.dumps(payload, ensure_ascii=False, default=str, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    payload["rejected_location"] = f"s3://trusted-zone/{key}"
    STRUCTURED_REJECTION_EVENTS.append(payload)
    print(f"Rejected evidence written to s3://trusted-zone/{key}")


def quarantine_invalid_required_rows(df: DataFrame, table_name: str, source_path: str, logical_date: str) -> tuple[DataFrame, int]:
    required_columns = required_columns_for_table(table_name)
    if not required_columns:
        return df, 0
    conditions = []
    for column_name in required_columns:
        field = df.schema[column_name]
        column_ref = F.col(column_name)
        if isinstance(field.dataType, StringType):
            conditions.append(column_ref.isNull() | (F.trim(column_ref) == ""))
        else:
            conditions.append(column_ref.isNull())
    invalid_condition = reduce(lambda left, right: left | right, conditions)
    invalid_df = df.where(invalid_condition)
    invalid_count = invalid_df.count()
    if invalid_count:
        sample_rows = [row.asDict(recursive=True) for row in invalid_df.limit(5).collect()]
        write_rejected_event(
            table_name,
            logical_date,
            {
                "reason": "missing_required_values",
                "source_file_path": source_path,
                "required_columns": required_columns,
                "rejected_record_count": invalid_count,
                "sample_records": sample_rows,
            },
        )
    return df.where(~invalid_condition), invalid_count


def validate_no_duplicate_keys(df: DataFrame, table_name: str) -> None:
    unique_keys = rules_for_table(table_name).get("unique_keys", [])
    if not unique_keys:
        return
    duplicate_count = df.groupBy(*unique_keys).count().where(F.col("count") > 1).limit(1).count()
    if duplicate_count:
        raise ValueError(f"{table_name} still contains duplicate keys after deduplication: {unique_keys}")


def add_governance_metadata(df: DataFrame, source_path: str, table_name: str) -> DataFrame:
    return (
        df.withColumn("source_system", F.lit("landing-zone"))
        .withColumn("ingestion_time", F.lit(TRUSTED_LOGICAL_DATE))
        .withColumn("source_file_path", F.lit(source_path))
        .withColumn("validation_status", F.lit("valid"))
        .withColumn("schema_version", F.lit(schema_version_for_table(table_name)))
    )


### Step 5 - Deduplication, ClickHouse DDL, and Trusted Write

After invalid rows are quarantined, we fill only sorting-key nulls, deduplicate using the same conservative rule as the DAG, generate a MergeTree DDL, and write with the Trusted ClickHouse service user.


In [7]:
def choose_sorting_keys(df: DataFrame, table_name: str) -> list[str]:
    sorting_keys = []
    for table_hint, candidate_keys in TABLE_SORTING_KEYS.items():
        if table_hint in table_name:
            sorting_keys = candidate_keys
            break
    final_keys = [key for key in sorting_keys if key in df.columns]
    if not final_keys and "id" in df.columns:
        final_keys = ["id"]
    if not final_keys and df.columns:
        final_keys = [df.columns[0]]
    return final_keys


def fill_sorting_key_nulls(df: DataFrame, sorting_keys: list[str]) -> DataFrame:
    dtype_lookup = dict(df.dtypes)
    for column_name in sorting_keys:
        column_type = dtype_lookup[column_name]
        if "string" in column_type:
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit("unknown")))
        elif any(token in column_type for token in ["int", "long", "short", "byte"]):
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit(0)))
        elif "double" in column_type or "float" in column_type:
            df = df.withColumn(column_name, F.coalesce(F.col(column_name), F.lit(0.0)))
    return df


def deduplicate_for_trusted_zone(df: DataFrame, sorting_keys: list[str], table_name: str) -> DataFrame:
    known_business_table = any(token in table_name for token in ["emission", "global_warming", "temperature_change", "tweet"])
    if known_business_table:
        return df.dropDuplicates()
    if sorting_keys:
        return df.dropDuplicates(subset=sorting_keys)
    return df.dropDuplicates()


def clickhouse_type_for_field(field) -> str:
    if isinstance(field.dataType, ArrayType):
        element_type = CLICKHOUSE_TYPE_MAP.get(type(field.dataType.elementType), "String")
        return f"Array({element_type})"
    if isinstance(field.dataType, DecimalType):
        return f"Decimal({field.dataType.precision},{field.dataType.scale})"
    return CLICKHOUSE_TYPE_MAP.get(type(field.dataType), "String")


def generate_clickhouse_ddl(df: DataFrame, table_name: str, sorting_keys: list[str], database_name: str) -> str:
    if not df.columns:
        raise ValueError(f"Cannot generate ClickHouse DDL for empty schema table {table_name}")
    ch_columns = []
    sorting_key_set = set(sorting_keys)
    for field in df.limit(0).schema.fields:
        ch_type = clickhouse_type_for_field(field)
        is_array = isinstance(field.dataType, ArrayType)
        if field.name in sorting_key_set or is_array:
            ch_columns.append(f"    `{field.name}` {ch_type}")
        else:
            ch_columns.append(f"    `{field.name}` Nullable({ch_type})")
    order_by_clause = ", ".join([f"`{key}`" for key in sorting_keys]) if sorting_keys else f"`{df.columns[0]}`"
    return (
        f"CREATE TABLE IF NOT EXISTS {database_name}.{table_name} (\n"
        f"{',\n'.join(ch_columns)}\n"
        ") ENGINE = MergeTree()\n"
        f"ORDER BY ({order_by_clause})"
    )


def load_dataframe_to_clickhouse_parallel(df: DataFrame, ddl_sql: str, target_table_name: str, database_name: str = "bi_analytics"):
    driver_client = clickhouse_connect.get_client(
        host="clickhouse",
        port=8123,
        username=os.getenv("CLICKHOUSE_TRUSTED_USER", os.getenv("CLICKHOUSE_USER", "analytics")),
        password=os.getenv("CLICKHOUSE_TRUSTED_PASSWORD", os.getenv("CLICKHOUSE_PASSWORD", "analytics_secret")),
        database=database_name,
    )
    driver_client.command(f"DROP TABLE IF EXISTS {database_name}.{target_table_name}")
    driver_client.command(ddl_sql)
    driver_client.close()
    column_names = df.columns
    partition_batches = df.rdd.getNumPartitions()
    print(f"Starting ClickHouse sync for '{target_table_name}' with {partition_batches} Spark partition batches.")

    def write_partition(rows_iter):
        worker_client = clickhouse_connect.get_client(
            host="clickhouse",
            port=8123,
            username=os.getenv("CLICKHOUSE_TRUSTED_USER", os.getenv("CLICKHOUSE_USER", "analytics")),
            password=os.getenv("CLICKHOUSE_TRUSTED_PASSWORD", os.getenv("CLICKHOUSE_PASSWORD", "analytics_secret")),
            database=database_name,
        )
        payload = [tuple(row) for row in rows_iter]
        if payload:
            print(f"[ClickHouse partition batch] target='{target_table_name}' rows={len(payload)}")
            worker_client.insert(table=target_table_name, data=payload, column_names=column_names)
        worker_client.close()
        return [len(payload)]

    total_inserted = df.rdd.mapPartitions(write_partition).sum()
    print(f"Parallel sync completed for '{target_table_name}'. Total rows ingested: {total_inserted}")


### Step 6 - Execute Cleaning One Dataset at a Time

This orchestration cell intentionally exposes every row-changing step. The resulting audit table can be used directly in the report to prove cleaning correctness and rejected/quarantine behavior.


In [8]:
def clean_structured_dataset_stepwise(s3_path: str, database_name: str = "bi_analytics") -> tuple[DataFrame, str, str]:
    table_name = normalize_table_name(s3_path)
    if not table_name:
        raise ValueError(f"Excluded or invalid structured path: {s3_path}")
    print(f"\nProcessing target: {table_name}")

    df = read_delta_or_parquet(spark, s3_path)
    raw_count = df.count()
    audit_step(table_name, "load_source", None, raw_count, s3_path)

    df = standardize_column_names(df)
    log_expected_schema_gaps(df, table_name)
    audit_step(table_name, "normalize_columns", raw_count, raw_count, f"columns={df.columns}")

    df = apply_table_specific_fixes(df, table_name)
    audit_step(table_name, "table_specific_fixes", raw_count, raw_count, "isolated non-row-changing fixes")

    before_count = df.count()
    df = drop_empty_rows(df)
    after_count = df.count()
    audit_step(table_name, "drop_empty_rows", before_count, after_count)

    df = normalize_string_columns(df, table_name)
    audit_step(table_name, "normalize_strings_and_null_tokens", after_count, after_count)

    df = mask_invalid_numeric_values(df)
    audit_step(table_name, "mask_nan_and_infinity", after_count, after_count)

    df = normalize_array_columns(df)
    audit_step(table_name, "normalize_array_like_columns", after_count, after_count)

    try:
        validate_required_columns(df, table_name)
        audit_step(table_name, "validate_required_columns", after_count, after_count, ", ".join(required_columns_for_table(table_name)) or "no required columns")
    except ValueError as exc:
        write_rejected_event(
            table_name,
            TRUSTED_LOGICAL_DATE,
            {
                "reason": "missing_required_columns",
                "source_file_path": s3_path,
                "required_columns": required_columns_for_table(table_name),
                "observed_columns": df.columns,
                "error": str(exc),
            },
        )
        raise

    df = apply_dataset_specific_rules(df, table_name)
    audit_step(table_name, "apply_dataset_specific_rules", after_count, after_count, f"schema_version={schema_version_for_table(table_name)}")

    before_quarantine = df.count()
    df, rejected_count = quarantine_invalid_required_rows(df, table_name, s3_path, TRUSTED_LOGICAL_DATE)
    after_quarantine = df.count()
    audit_step(table_name, "quarantine_invalid_required_rows", before_quarantine, after_quarantine, f"rejected={rejected_count}")

    sorting_keys = choose_sorting_keys(df, table_name)
    df = fill_sorting_key_nulls(df, sorting_keys)
    audit_step(table_name, "fill_sorting_key_nulls", after_quarantine, after_quarantine, f"sorting_keys={sorting_keys}")

    before_dedup = df.count()
    df = deduplicate_for_trusted_zone(df, sorting_keys, table_name)
    after_dedup = df.count()
    audit_step(table_name, "deduplicate", before_dedup, after_dedup)

    validate_no_duplicate_keys(df, table_name)
    audit_step(table_name, "validate_unique_keys", after_dedup, after_dedup, ", ".join(rules_for_table(table_name).get("unique_keys", [])) or "not configured")

    df = add_governance_metadata(df, s3_path, table_name)
    final_count = df.count()
    audit_step(table_name, "append_governance_metadata", after_dedup, final_count, "source_system, ingestion_time, source_file_path, validation_status, schema_version")

    ddl_sql = generate_clickhouse_ddl(df, table_name, sorting_keys, database_name)
    return df, ddl_sql, table_name


def process_structured_dataset(s3_path: str, database_name: str = "bi_analytics") -> dict:
    try:
        cleaned_df, clickhouse_ddl, table_name = clean_structured_dataset_stepwise(s3_path, database_name=database_name)
        load_dataframe_to_clickhouse_parallel(cleaned_df, clickhouse_ddl, target_table_name=table_name, database_name=database_name)
        return {"table": table_name, "status": "success", "rows": cleaned_df.count()}
    except Exception as exc:
        table_name = normalize_table_name(s3_path)
        print(f"Synchronizer halted processing target table '{table_name}'. Error: {exc}")
        return {"table": table_name, "status": "failed", "error": str(exc)}


def run_structured_cleaning_pipeline(dataset_paths: list[str]) -> list[dict]:
    print("Starting step-by-step Structured Trusted cleaning pipeline")
    results = []
    total_batches = len(dataset_paths)
    for batch_no, s3_path in enumerate(dataset_paths, start=1):
        print(f"\n[structured cleaning batch {batch_no}/{total_batches}] source={s3_path}")
        result = process_structured_dataset(s3_path)
        results.append(result)
        print(f"[structured cleaning batch {batch_no}/{total_batches}] status={result.get('status')} table={result.get('table')} rows={result.get('rows', 'n/a')}")
    successful = sum(1 for result in results if result["status"] == "success")
    print(f"\nStructured cleaning pipeline completed: {successful}/{len(results)} tables synced successfully.")
    return results


pipeline_results = run_structured_cleaning_pipeline(valid_delta_paths)
display(pd.DataFrame(pipeline_results))


Starting step-by-step Structured Trusted cleaning pipeline

[structured cleaning batch 1/4] source=s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975

Processing target: co2_emission_by_vehicles
Reading s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975 as delta
[co2_emission_by_vehicles] load_source: None -> 7385. s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975
Column normalization mapping: {'Make': 'make', 'Model': 'model', 'Vehicle Class': 'vehicle_class', 'Engine Size(L)': 'engine_sizel', 'Cylinders': 'cylinders', 'Transmission': 'transmission', 'Fuel Type': 'fuel_type', 'Fuel Consumption City (L/100 km)': 'fuel_consumption_city_l_100_km', 'Fuel Consumption Hwy (L/100 km)': 'fuel_consumption_hwy_l_100_km', 'Fuel Consumption Comb (L/100 km)': 'fuel_consumption_comb_l_100_km', 'Fuel Consumption Comb (mpg)': 'fuel_consumption_comb_mpg', 'CO2 Emissions(g/km)': 'co2_emission

,table,status,rows
0,co2_emission_by_vehicles,success,5988
1,global_warming_dataset,success,100000
2,natural_disaster_tweets,success,107593
3,temperature_change,success,241893


### Step 7 - Cleaning Audit and Rejected Evidence Summary

The audit table shows row-count movement per step. Rejected evidence lists any structured quarantine events written by the notebook run.


In [9]:
display(pd.DataFrame(STRUCTURED_CLEANING_AUDIT))

if STRUCTURED_REJECTION_EVENTS:
    display(pd.DataFrame(STRUCTURED_REJECTION_EVENTS))
else:
    print("No structured rejected rows/events were produced in this notebook run.")


,table,step,before_rows,after_rows,detail
0,co2_emission_by_vehicles,load_source,NaN,7385,s3a://landing-zone/persistent-landing/structur...
1,co2_emission_by_vehicles,normalize_columns,7385.0,7385,"columns=['make', 'model', 'vehicle_class', 'en..."
2,co2_emission_by_vehicles,table_specific_fixes,7385.0,7385,isolated non-row-changing fixes
3,co2_emission_by_vehicles,drop_empty_rows,7385.0,7385,
4,co2_emission_by_vehicles,normalize_strings_and_null_tokens,7385.0,7385,
5,co2_emission_by_vehicles,mask_nan_and_infinity,7385.0,7385,
6,co2_emission_by_vehicles,normalize_array_like_columns,7385.0,7385,
7,co2_emission_by_vehicles,validate_required_columns,7385.0,7385,"make, model, vehicle_class"
8,co2_emission_by_vehicles,apply_dataset_specific_rules,7385.0,7385,schema_version=co2_emission_by_vehicles_v1
9,co2_emission_by_vehicles,quarantine_invalid_required_rows,7385.0,7385,rejected=0


,zone,domain,dataset_name,validation_status,schema_version,rejected_at,reason,source_file_path,required_columns,rejected_record_count,sample_records,rejected_location
0,trusted,structured,natural_disaster_tweets,rejected,natural_disaster_tweets_v1,2026-06-06T17:05:29.309639+00:00,missing_required_values,s3a://landing-zone/persistent-landing/structur...,[id],20000,"[{'id': None, 'tweet_text': '#turkeyearthquake...",s3://trusted-zone/rejected/structured/natural_...


### Post-Cleaning Data Validation
This testing module connects to the final ClickHouse physical data warehouse layer to verify:
1. **Physical Storage Metrics**: Check whether the stored row count has severely shrunk/expanded, and monitor disk space utilization.
2. **Data Sampling**: Visually verify that composite primary keys and nested arrays meet ClickHouse's business requirements.

In [10]:
import pandas as pd


def get_clickhouse_client(database_name: str = "bi_analytics"):
    """Create a ClickHouse client for validation queries."""
    return clickhouse_connect.get_client(
        host="clickhouse",
        port=8123,
        username="analytics",
        password="analytics_secret",
        database=database_name,
    )


def print_storage_metrics(client, database_name: str = "bi_analytics") -> None:
    """Print physical row counts and storage size for active ClickHouse parts."""
    metrics_query = f"""
    SELECT
        table AS table_name,
        sum(rows) AS total_physical_rows,
        formatReadableSize(sum(bytes_on_disk)) AS disk_usage_size
    FROM system.parts
    WHERE database = '{database_name}' AND active = 1
    GROUP BY table
    ORDER BY table
    """
    print("Data Warehouse Storage Metrics (ClickHouse)")
    for row in client.query(metrics_query).result_rows:
        print(f"Table: {row[0]:<30} | Rows: {row[1]:<10} | Disk Storage: {row[2]}")


def check_temperature_unit_encoding(client, database_name: str = "bi_analytics") -> None:
    """Verify that temperature_change.unit no longer contains mojibake markers."""
    bad_marker_1 = chr(0x00e2)
    bad_marker_2 = chr(0x00c2)
    unit_check_query = f"""
    SELECT unit, count()
    FROM {database_name}.temperature_change
    WHERE position(unit, '{bad_marker_1}') > 0 OR position(unit, '{bad_marker_2}') > 0
    GROUP BY unit
    """
    bad_units = client.query(unit_check_query).result_rows
    if bad_units:
        print("WARNING: Remaining mojibake unit values detected:", bad_units)
    else:
        print("Unit encoding check passed: no mojibake values found in temperature_change.unit.")


def preview_target_tables(client, target_tables: list[str], database_name: str = "bi_analytics", sample_size: int = 3) -> None:
    """Display small untruncated samples from each target table."""
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", 1000)

    print("Target Table Data Sampling Preview (Top 3 Rows - Untruncated)")
    for table in target_tables:
        try:
            headers_query = f"SELECT name FROM system.columns WHERE database = '{database_name}' AND table = '{table}' ORDER BY position"
            headers = [row[0] for row in client.query(headers_query).result_rows]
            data_res = client.query(f"SELECT * FROM {database_name}.{table} LIMIT {sample_size}")

            print(f"\nTable: {table}")
            print("-" * 80)
            if not data_res.result_rows:
                print("  Empty: No data available in ClickHouse.")
                continue

            pdf = pd.DataFrame(data_res.result_rows, columns=headers)
            for column_name in pdf.columns:
                pdf[column_name] = pdf[column_name].apply(lambda value: list(value) if isinstance(value, (list, tuple)) else value)
            display(pdf)

            if table == "temperature_change" and "unit" in headers:
                check_temperature_unit_encoding(client, database_name=database_name)

        except Exception as exc:
            print(f"Could not preview table '{table}': {exc}")


client = get_clickhouse_client()
try:
    print_storage_metrics(client)
    preview_target_tables(
        client,
        ["natural_disaster_tweets", "co2_emission_by_vehicles", "temperature_change", "global_warming_dataset"],
    )
finally:
    client.close()


Data Warehouse Storage Metrics (ClickHouse)
Table: co2_emission_by_vehicles       | Rows: 5988       | Disk Storage: 135.87 KiB
Table: global_warming_dataset         | Rows: 100000     | Disk Storage: 18.19 MiB
Table: natural_disaster_tweets        | Rows: 107593     | Disk Storage: 10.34 MiB
Table: temperature_change             | Rows: 241893     | Disk Storage: 2.78 MiB
Target Table Data Sampling Preview (Top 3 Rows - Untruncated)

Table: natural_disaster_tweets
--------------------------------------------------------------------------------


,id,tweet_text,disaster_type,hashtags,emojis,source_system,ingestion_time,source_file_path,validation_status,schema_version
0,1.00113669658963149e18,"flash floods struck a maryland city on sunday, washing out streets and tossing cars like bath toys.",flood,[],[],landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/natural_disaster_tweets_1780764519520,valid,natural_disaster_tweets_v1
1,1.00113837492357939e18,"catastrophic flooding slams ellicott city, maryland; water rescues reported - the weather channel via @googlenews",flood,[],[],landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/natural_disaster_tweets_1780764519520,valid,natural_disaster_tweets_v1
2,1.00113964402369331e18,i am inordinately pleased that essentially all of the replies to this tweet are people calling out fox for using @dierobinsondie s video after being explicitly denied permission to do so. (plus some prayers &amp; well-wishes for the folks in the flood.),flood,[],[],landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/natural_disaster_tweets_1780764519520,valid,natural_disaster_tweets_v1



Table: co2_emission_by_vehicles
--------------------------------------------------------------------------------


,make,model,vehicle_class,engine_sizel,cylinders,transmission,fuel_type,fuel_consumption_city_l_100_km,fuel_consumption_hwy_l_100_km,fuel_consumption_comb_l_100_km,fuel_consumption_comb_mpg,co2_emissionsg_km,source_system,ingestion_time,source_file_path,validation_status,schema_version
0,acura,ilx,compact,2.4,4,am8,z,9.3,6.6,8.1,35,189,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975,valid,co2_emission_by_vehicles_v1
1,acura,ilx,compact,2.4,4,m6,z,11.2,7.7,9.6,29,221,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975,valid,co2_emission_by_vehicles_v1
2,acura,ilx,compact,2.4,4,m6,z,10.8,7.4,9.3,30,214,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/co2-emission-by-vehicles_1780764135975,valid,co2_emission_by_vehicles_v1



Table: temperature_change
--------------------------------------------------------------------------------


,domain_code,domain,area_code_m49,area,element_code,element,months_code,months,year_code,year,unit,value,flag,flag_description,source_system,ingestion_time,source_file_path,validation_status,schema_version
0,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1970,1970,°C,1.412,e,estimated value,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/temperature_change_1780764519800,valid,temperature_change_v1
1,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1971,1971,°C,1.518,e,estimated value,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/temperature_change_1780764519800,valid,temperature_change_v1
2,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1974,1974,°C,1.032,e,estimated value,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/temperature_change_1780764519800,valid,temperature_change_v1


Unit encoding check passed: no mojibake values found in temperature_change.unit.

Table: global_warming_dataset
--------------------------------------------------------------------------------


,country,year,temperature_anomaly,co2_emissions,population,forest_area,gdp,renewable_energy_usage,methane_emissions,sea_level_rise,arctic_ice_extent,urbanization,deforestation_rate,extreme_weather_events,average_rainfall,solar_energy_potential,waste_management,per_capita_emissions,industrial_activity,air_pollution_index,biodiversity_index,ocean_acidification,fossil_fuel_usage,energy_consumption_per_capita,policy_score,average_temperature,source_system,ingestion_time,source_file_path,validation_status,schema_version
0,country_1,1904,0.821844,3.801896e+08,3.636077e+08,5.460839,9.631346e+12,81.527857,4.802553e+06,20.112365,4.217669,18.481736,3.964870,33,4196.254647,1862.027591,64.601667,18.189224,79.915463,131.523195,64.003342,8.165741,65.407002,2315.420801,17.278762,-5.657269,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/global_warming_dataset_1780764136045,valid,global_warming_dataset_v1
1,country_1,1907,-0.490971,8.182223e+08,7.296803e+08,47.399085,6.671216e+12,89.007450,4.931721e+06,17.155683,2.131775,34.894075,0.587480,36,1507.208899,867.291139,71.992918,11.298258,32.759118,94.636642,0.194610,8.076849,85.683252,2873.453304,73.814180,39.315752,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/global_warming_dataset_1780764136045,valid,global_warming_dataset_v1
2,country_1,1908,1.453433,2.921778e+08,1.016159e+07,16.411747,7.217582e+12,35.543250,1.843125e+06,29.233273,7.223588,87.676863,0.568711,35,1076.945205,1544.353842,47.082919,5.512314,19.923577,101.707250,88.758200,7.582983,40.817512,3053.640049,5.319145,-4.950758,landing-zone,2026-06-06T17:05:29.309639+00:00,s3a://landing-zone/persistent-landing/structured/global_warming_dataset_1780764136045,valid,global_warming_dataset_v1


In [11]:
spark.stop()